In [1]:
# !export CUDA_VISIBLE_DEVICES=5

In [2]:
import os
# os.environ["CUDA_VISIBLE_DEVICES"] = "5"

In [3]:
# import tensorflow as tf
from transformers import BertTokenizerFast, BertForSequenceClassification
import pandas as pd 
import torch
from sklearn.model_selection import train_test_split
import numpy as np
import re

c:\Users\rachu\MTECH\SEMESTER 4\Individual Project\Phase 1 Code\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
if torch.cuda.is_available():
    print("CUDA is available. Number of GPUs:", torch.cuda.device_count())
    print("CUDA device name:", torch.cuda.get_device_name(0))
else:
    print("CUDA is not available.")
# torch.cuda.set_device(torch.device("cuda:0"))
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

CUDA is not available.


c:\Users\rachu\MTECH\SEMESTER 4\Individual Project\Phase 1 Code\.venv\Lib\site-packages\torch\cuda\__init__.py:129: UserWarning: CUDA initialization: Unexpected error from cudaGetDeviceCount(). Did you run some cuda functions before calling NumCudaDevices() that might have already set an error? Error 1: invalid argument (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\c10\cuda\CUDAFunctions.cpp:109.)
  return torch._C._cuda_getDeviceCount() > 0


In [5]:
tokenizer = BertTokenizerFast.from_pretrained("google-bert/bert-base-uncased")
model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=5).to(device)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6352.10it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoi

In [6]:
# optimizer = tf.keras.optimizers.Adam(learning_rate=5e-5)
# loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
# #metrics = [tf.keras.metrics.SparseCategoricalAccuracy()]

In [7]:
# strategy = tf.distribute.MirroredStrategy()
if torch.cuda.is_available():
        print(f"LOG: Torch allocated Memory: \
                {torch.cuda.memory_allocated() / 1024 ** 2:.2f} MB", flush=True)
        print(f"LOG: Torch cached Memory: \
                {torch.cuda.memory_reserved() / 1024 ** 2:.2f} MB", flush=True)

# if tf.config.experimental.list_physical_devices('GPU'):
#     zero_info = tf.config.experimental.get_memory_info('GPU:0')
#     print(f"LOG: TensorFlow, 0: Current memory usage: \
#             {zero_info['current'] / 1024 ** 2:.2f} MB", flush=True)
#     print(f"LOG: TensorFlow, 0: Peak memory usage: \
#             {zero_info['peak'] / 1024 ** 2:.2f} MB", flush=True)

# model.compile(optimizer=Adam(3e-5))  # No loss argument!

# model.compile(optimizer=optimizer, loss=loss, metrics=metrics)

In [8]:
#Now working with the text

In [9]:
import os
import pandas as pd

topic_id = 0

# Check paths for both VS Code notebook folder execution and workspace root execution
possible_paths = [
    f'../Topic_Files/topic_{topic_id}.csv',
    f'Base_Main/Topic_Files/topic_{topic_id}.csv',
    os.path.join(os.getcwd(), '..', 'Topic_Files', f'topic_{topic_id}.csv'),
    os.path.join(os.getcwd(), 'Base_Main', 'Topic_Files', f'topic_{topic_id}.csv')
]

topic_path = None
for p in possible_paths:
    if os.path.exists(p):
        topic_path = p
        break

if topic_path is None:
    raise FileNotFoundError(f"Could not locate topic_{topic_id}.csv in expected paths.")

print(f"Loading Topic {topic_id} from: {os.path.abspath(topic_path)}")
df = pd.read_csv(topic_path)
print(f"Loaded Topic {topic_id} dataset: {len(df)} records")
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)

Loading Topic 0 from: c:\Users\rachu\MTECH\SEMESTER 4\Individual Project\Phase 1 Code\Base_Main\Topic_Files\topic_0.csv
Loaded Topic 0 dataset: 1033 records


In [10]:
max_length = 512
def tokenize_function(text):
    return tokenizer(text, padding= "max_length", truncation=True)


In [11]:
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

In [12]:
max_length = 512
train_encodings = tokenizer(
    train_df['Combined_Text'].tolist(), 
    truncation=True, 
    padding=True, 
    max_length=max_length
)
val_encodings = tokenizer(
    val_df['Combined_Text'].tolist(), 
    truncation=True, 
    padding=True, 
    max_length=max_length
)

In [13]:
train_inputs = {key: torch.tensor(val) for key, val in train_encodings.items()}
val_inputs = {key: torch.tensor(val) for key, val in val_encodings.items()}
train_labels = torch.tensor(train_df['Priority'].values - 1)  # Adjust labels to start from 0
val_labels = torch.tensor(val_df['Priority'].values - 1) 

In [14]:
#train_dataset = tf.data.Dataset.from_tensor_slices((train_encodings, train_labels)).shuffle(len(train_df)).batch(15)
#val_dataset = tf.data.Dataset.from_tensor_slices((val_encodings, val_labels)).batch(15)


In [15]:
# Create TensorFlow datasets
#train_dataset = tf.data.Dataset.from_tensor_slices((
#    dict(train_encodings),
#    train_df['Priority']
#)).shuffle(len(train_df)).batch(15)

#val_dataset = tf.data.Dataset.from_tensor_slices((
#    dict(val_encodings),
#    val_df['Priority']
#)).batch(15)

In [16]:
from torch.utils.data import Dataset, DataLoader
class CustomDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        label = self.labels[idx]
        return item, label

    def __len__(self):
        return len(self.labels)

# Create DataLoader
train_dataset = CustomDataset(train_encodings, train_labels)
val_dataset = CustomDataset(val_encodings, val_labels)

In [17]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, pin_memory=True)


In [18]:
#input_ids = inputs['input_ids']
#attention_mask = inputs['attention_mask']

In [19]:
import os
import torch
from transformers import BertTokenizerFast, BertForSequenceClassification
import pandas as pd
from sklearn.model_selection import train_test_split
from torch.optim import Adam
from torch.cuda.amp import GradScaler, autocast
from torch.utils.data import Dataset, DataLoader


In [ ]:
import torch
from torch.optim import Adam
import torch.nn.functional as F
from torch.cuda.amp import GradScaler, autocast
from torch.utils.data import DataLoader

num_epochs = 1
optimizer = Adam(model.parameters(), lr=1e-5)
loss_fn = torch.nn.CrossEntropyLoss()
scaler = GradScaler()

print(f"Starting training on Topic {topic_id} ({len(train_loader)} batches)...")

# Training loop
for epoch in range(num_epochs):
    model.train()
    total_loss = 0.0
    for step, (batch, labels) in enumerate(train_loader):
        inputs = {key: val.to(device) for key, val in batch.items()}
        labels = labels.to(device)
        
        optimizer.zero_grad()

        with autocast():
            outputs = model(**inputs)
            logits = outputs.logits
            logits = logits.view(-1, 5)

            loss = loss_fn(logits, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        if (step + 1) % 5 == 0 or (step + 1) == len(train_loader):
            print(f"Epoch [{epoch + 1}/{num_epochs}], Step [{step + 1}/{len(train_loader)}], Loss: {total_loss / (step + 1):.4f}")

print(f"Training for Topic {topic_id} finished successfully!")

C:\Users\rachu\AppData\Local\Temp\ipykernel_29344\2499131955.py:10: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
c:\Users\rachu\MTECH\SEMESTER 4\Individual Project\Phase 1 Code\.venv\Lib\site-packages\torch\amp\grad_scaler.py:132: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  warnings.warn(
C:\Users\rachu\AppData\Local\Temp\ipykernel_29344\2499131955.py:24: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
c:\Users\rachu\MTECH\SEMESTER 4\Individual Project\Phase 1 Code\.venv\Lib\site-packages\torch\amp\autocast_mode.py:266: UserWarning: User provided device_type of 'cuda', but CUDA is not available. Disabling
  warnings.warn(


Starting training on Topic 0 (26 batches)...
Epoch [1/1], Step [5/26], Loss: 1.0993


In [ ]:
import joblib
import os

# Save in BERTTrained directory
if os.path.exists('BERTTrained') or not os.path.exists('Base_Main'):
    extract_dir = 'BERTTrained'
else:
    extract_dir = 'Base_Main/TrainingBERTClassifiers/BERTTrained'

os.makedirs(extract_dir, exist_ok=True)
topic_pred_file = os.path.join(extract_dir, f'topic_pred_{topic_id}.pkl')
joblib.dump(model, topic_pred_file)
print(f"Saved trained model to: {os.path.abspath(topic_pred_file)}")